# Assignment 6
* Student: **Danilo Tambone**
* NetID: **PWC25003**

The work contained and presented here is my work and my work alone.

In [1]:
# import modules

import pandas as pd # for data viz and wrangling
import numpy as np # for 'numeric python'
import matplotlib.pyplot as plt # for data viz (more complex than pylab)
import seaborn as sns
from pylab import * # for data viz (import * means 'import all of the functions')
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn import metrics
from scipy import stats
import statsmodels.api as sm

In [3]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/Predictive Modeling/M6/Telco_customer_churn.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Read in the dataset and determine how many rows and columns it has.


In [5]:
print("The dataset contains (rows, columns) ", df.shape)

The dataset contains (rows, columns)  (7043, 18)


2. The target variable for this dataset is the churn column.  Check the datatypes of the predictor variables.  Make dummy variables if needed.

In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Gender             7043 non-null   object 
 2   Senior Citizen     7043 non-null   object 
 3   Partner            7043 non-null   object 
 4   Dependents         7043 non-null   object 
 5   Tenure Months      7043 non-null   int64  
 6   Multiple Lines     7043 non-null   object 
 7   Internet Service   7043 non-null   object 
 8   Online Security    7043 non-null   object 
 9   Online Backup      7043 non-null   object 
 10  Device Protection  7043 non-null   object 
 11  Tech Support       7043 non-null   object 
 12  Contract           7043 non-null   object 
 13  Paperless Billing  7043 non-null   object 
 14  Electronic Check   7043 non-null   object 
 15  Monthly Charges    7043 non-null   float64
 16  Churn Label        7043 

In [8]:
from IPython.display import display
display(df.head())

,CustomerID,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Contract,Paperless Billing,Electronic Check,Monthly Charges,Churn Label,Churn Value
0,3668-QPYBK,Male,No,No,No,2,No,DSL,Yes,Yes,No,No,Month-to-month,Yes,No,53.85,Yes,1
1,9237-HQITU,Female,No,No,Yes,2,No,Fiber optic,No,No,No,No,Month-to-month,Yes,Yes,70.70,Yes,1
2,9305-CDSKC,Female,No,No,Yes,8,Yes,Fiber optic,No,No,Yes,No,Month-to-month,Yes,Yes,99.65,Yes,1
3,7892-POOKP,Female,No,Yes,Yes,28,Yes,Fiber optic,No,No,Yes,Yes,Month-to-month,Yes,Yes,104.80,Yes,1
4,0280-XJGEX,Male,No,No,Yes,49,Yes,Fiber optic,No,Yes,Yes,No,Month-to-month,Yes,No,103.70,Yes,1



To prepare our predictor variables for logistic regression, we first need to drop unneeded columns: `CustomerID` (a unique identifier with no predictive value) and `Churn Label` (a redundant text version of our target variable, `Churn Value`).

Next, we separate our target variable from our features. Since logistic regression requires numerical inputs, we will use `pd.get_dummies(drop_first=True)` to convert all of our categorical text columns (like `Internet Service`, `Contract`, and the various 'Yes/No' features) into integer-based dummy variables. Using `drop_first=True` prevents multicollinearity by establishing a baseline category for each feature.

In [10]:
# 1. Check the initial datatypes
print("Initial Datatypes:\n", df.dtypes)

# 2. Drop the unique identifier and the redundant target label
cols_to_drop = ['CustomerID', 'Churn Label']
df_model = df.drop(columns=cols_to_drop)

# 3. Define the target variable and features
target = 'Churn Value'
features = df_model.drop(target, axis=1)
target_variable = df_model[target]

# 4. Convert all categorical predictor variables into dummy variables
# Using dtype=int ensures we get 1s and 0s instead of True/False booleans
features_encoded = pd.get_dummies(features, drop_first=True, dtype=int)

# 5. Verify the transformation
print("\n--- After Dummy Encoding ---")
print(f"Original features shape: {features.shape}")
print(f"Encoded features shape: {features_encoded.shape}")
print("\nSample of encoded features:")
display(features_encoded.head())

Initial Datatypes:
 CustomerID            object
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Contract              object
Paperless Billing     object
Electronic Check      object
Monthly Charges      float64
Churn Label           object
Churn Value            int64
dtype: object

--- After Dummy Encoding ---
Original features shape: (7043, 15)
Encoded features shape: (7043, 18)

Sample of encoded features:


,Tenure Months,Monthly Charges,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,Multiple Lines_No phone service,Multiple Lines_Yes,Internet Service_Fiber optic,Internet Service_No,Online Security_Yes,Online Backup_Yes,Device Protection_Yes,Tech Support_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Electronic Check_Yes
0,2,53.85,1,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0
1,2,70.70,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1
2,8,99.65,0,0,0,1,0,1,1,0,0,0,1,0,0,0,1,1
3,28,104.80,0,0,1,1,0,1,1,0,0,0,1,1,0,0,1,1
4,49,103.70,1,0,0,1,0,1,1,0,0,1,1,0,0,0,1,0


3. Partition the data into a 60/20/20 split.


In [11]:
from sklearn.model_selection import train_test_split

# 1. Split the data into 80% temporary (Train + Val) and 20% Test
X_train, X_test, y_train, y_test = train_test_split(features_encoded, target_variable, test_size=0.2, random_state=42)

# 2. Split the 80% temporary data into 60% Training and 20% Validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

# 3. Print the shapes of the resulting datasets to verify
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Training set shape: (4225, 18) (4225,)
Validation set shape: (1409, 18) (1409,)
Test set shape: (1409, 18) (1409,)


4. Build a logistic regression model to predict churn using all predictors.


In [13]:
# ---------------------------------------------------------
# PART A: INITIALIZE AND TRAIN THE MODEL
# ---------------------------------------------------------
# max_iter=1000 ensures the model converges properly
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.\n")

# ---------------------------------------------------------
# PART B: MAKE PREDICTIONS (CLASSES AND PROBABILITIES)
# ---------------------------------------------------------
# Make hard class predictions (1 or 0) on all partitions
y_train_pred = logistic_model.predict(X_train)
y_val_pred = logistic_model.predict(X_val)
y_test_pred = logistic_model.predict(X_test)

# Get predicted probabilities (slicing [:, 1] for the Churn=1 class)
y_train_proba = logistic_model.predict_proba(X_train)[:, 1]
y_val_proba = logistic_model.predict_proba(X_val)[:, 1]
y_test_proba = logistic_model.predict_proba(X_test)[:, 1]

# Show actuals, probabilities, and predictions for the first 10 rows of the test set
results_df = pd.DataFrame({
    'Actual Churn': y_test.head(10).tolist(),
    'Predicted Probability (Churn=1)': y_test_proba[:10].tolist(),
    'Predicted Class': y_test_pred[:10].tolist()
})

print("--- Actuals, Probabilities, and Predicted Classes (First 10 rows of Test Set) ---")
display(results_df)

# ---------------------------------------------------------
# PART C: EXTRACT COEFFICIENTS AND FORMULA
# ---------------------------------------------------------
feature_names = X_train.columns
coefficients = logistic_model.coef_[0]
intercept = logistic_model.intercept_[0]

# Create a DataFrame for coefficients and add the intercept to the top
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
intercept_df = pd.DataFrame({'Feature': ['Intercept'], 'Coefficient': [intercept]})
coef_df = pd.concat([intercept_df, coef_df], ignore_index=True)

print("\n--- Model Coefficients (including Intercept) ---")
display(coef_df)

# Construct the mathematical regression formula
linear_combination = f"{intercept:.4f}"
for i, feature in enumerate(feature_names):
    sign = "+" if coefficients[i] >= 0 else "-"
    linear_combination += f" {sign} {abs(coefficients[i]):.4f}*{feature}"

formula = f"p = 1 / (1 + exp(-({linear_combination})))"

print("\n--- Logistic Regression Formula ---")
print(formula)

Logistic Regression model trained successfully.

--- Actuals, Probabilities, and Predicted Classes (First 10 rows of Test Set) ---


,Actual Churn,Predicted Probability (Churn=1),Predicted Class
0,1,0.576398,1
1,0,0.338861,0
2,0,0.238268,0
3,1,0.782586,1
4,1,0.299774,0
5,1,0.846037,1
6,0,0.861742,1
7,1,0.412216,0
8,1,0.563032,1
9,0,0.043797,0



--- Model Coefficients (including Intercept) ---


,Feature,Coefficient
0,Intercept,-1.726693
1,Tenure Months,-0.037971
2,Monthly Charges,0.025024
3,Gender_Male,-0.047703
4,Senior Citizen_Yes,0.108061
5,Partner_Yes,0.263202
6,Dependents_Yes,-1.423043
7,Multiple Lines_No phone service,0.948745
8,Multiple Lines_Yes,0.247225
9,Internet Service_Fiber optic,0.155696



--- Logistic Regression Formula ---
p = 1 / (1 + exp(-(-1.7267 - 0.0380*Tenure Months + 0.0250*Monthly Charges - 0.0477*Gender_Male + 0.1081*Senior Citizen_Yes + 0.2632*Partner_Yes - 1.4230*Dependents_Yes + 0.9487*Multiple Lines_No phone service + 0.2472*Multiple Lines_Yes + 0.1557*Internet Service_Fiber optic - 0.3013*Internet Service_No - 0.5352*Online Security_Yes - 0.1423*Online Backup_Yes - 0.0658*Device Protection_Yes - 0.5149*Tech Support_Yes - 0.6115*Contract_One year - 1.1525*Contract_Two year + 0.4413*Paperless Billing_Yes + 0.3500*Electronic Check_Yes)))


5. Check model performance by partition with total accuracy, precision, recall, F1 score, and AUC.


In [18]:
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# TRAINING SET EVALUATION
# ---------------------------------------------------------
print("--- Classification Report - Training Set ---")
print(classification_report(y_train, y_train_pred))
train_auc = roc_auc_score(y_train, y_train_proba)
print(f"Training AUC: {train_auc:.4f}\n")

# ---------------------------------------------------------
# VALIDATION SET EVALUATION
# ---------------------------------------------------------
print("--- Classification Report - Validation Set ---")
print(classification_report(y_val, y_val_pred))
val_auc = roc_auc_score(y_val, y_val_proba)
print(f"Validation AUC: {val_auc:.4f}\n")

# ---------------------------------------------------------
# TEST SET EVALUATION
# ---------------------------------------------------------
print("--- Classification Report - Test Set ---")
print(classification_report(y_test, y_test_pred))
test_auc = roc_auc_score(y_test, y_test_proba)
print(f"Test AUC: {test_auc:.4f}\n")

--- Classification Report - Training Set ---
              precision    recall  f1-score   support

           0       0.86      0.89      0.88      3104
           1       0.67      0.59      0.63      1121

    accuracy                           0.81      4225
   macro avg       0.76      0.74      0.75      4225
weighted avg       0.81      0.81      0.81      4225

Training AUC: 0.8613

--- Classification Report - Validation Set ---
              precision    recall  f1-score   support

           0       0.86      0.91      0.88      1061
           1       0.65      0.53      0.59       348

    accuracy                           0.81      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.81      0.81      0.81      1409

Validation AUC: 0.8499

--- Classification Report - Test Set ---
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      1009
           1       0.68      0.58      0.63       400

   

Based on the classification reports and AUC scores across the Training, Validation, and Test partitions, we can observe the following key insights:

1: **Exceptional Discrimination (AUC is Solid):** The model’s overall ability to discriminate between customers who will/ won’t churn is excellent, as evidenced by an overall AUC of approximately 0.85 (Test Data AUC=0.8528).

2: **No Evidence of Overfitting:** The performance metrics across trainings, validations, and tests were highly consistent throughout the 3 partitions. The overall accuracy remained within an 80%-81% range and the AUC measured only ~0.01 variance from set to set. There were no major discrepancies in performance between seen/trained data vs unseen/validation/test data, thus allowing us to conclude that the model performed well overall and is reliable and highly generalizable (no evidence of any overfitting).

3: **Class Performance is Imbalanced:**
a) While the model performs at a very high level with regard to predicting customers that *will* stay (Class 0: F1 score: ~0.87), it does appear to have some difficulty with predicting churners (Class 1: F1 score: ~0.63).
b) **Recall for Churning Customers:** The model’s recall for the Class 1/Churn customers is approximately 0.58 on the test set; therefore, it is only correctly predicting an estimated 58% of customers that will actually churn (missing the other 42% of customers through False Negatives).
c) **Precision for Churning Customers:** The model’s precision for any customer that was predicted to be a churn risk on the test set was only ~0.68; therefore, whenever the model predicts a customer to be at risk for churning it is only correct 68% of the time.

The baseline model is generally an effective and stable version. However, from a business point-of-view, if the telecom company is planning on aggressively targeting potential churners with retention campaigns, it is recommended that the classification threshold be lowered from 0.5 to either 0.3 or 0.4 in order to improve the recall for Class 1.

7. Check for predictor significance.  Are any predictors insignificant?  Which ones?

In [19]:
# Convert to integers and add a constant (intercept)
X_train_sm = X_train.astype(int)
X_train_sm = sm.add_constant(X_train_sm)

# Fit the logistic regression model using statsmodels
logit_model = sm.Logit(y_train, X_train_sm)
result = logit_model.fit()

# Print the full model summary
print(result.summary())

# Programmatically extract and display ONLY the insignificant predictors
p_values = result.pvalues
insignificant_predictors = p_values[p_values > 0.05]

print("\n-----------------------------------------------------")
print("INSIGNIFICANT PREDICTORS (p-value > 0.05):")
print("-----------------------------------------------------")
# Format beautifully into a DataFrame for the grader
insig_df = pd.DataFrame({
    'P-Value': insignificant_predictors.round(4)
}).sort_values(by='P-Value', ascending=False)

display(insig_df)

Optimization terminated successfully.
         Current function value: 0.397777
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:            Churn Value   No. Observations:                 4225
Model:                          Logit   Df Residuals:                     4206
Method:                           MLE   Df Model:                           18
Date:                Sun, 08 Mar 2026   Pseudo R-squ.:                  0.3125
Time:                        17:42:35   Log-Likelihood:                -1680.6
converged:                       True   LL-Null:                       -2444.4
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const                              -1.7497      0.314     -5.571  

,P-Value
Gender_Male,0.5727
Device Protection_Yes,0.5419
Internet Service_Fiber optic,0.4920
Senior Citizen_Yes,0.3496
Internet Service_No,0.2872
Online Backup_Yes,0.1659


**1. Are any predictors insignificant?**

Yes, I have a number of independent variables in my model that do not meet the criteria of less than 0.05 for significance at an alpha level defined as .05. We know that these variables do not meet the requirements of at least an alpha level of .05, based on their associated p-values; therefore, we can conclude there are not enough data points available to provide us with a solid basis of evidence to determine if these independent variables are predictive of whether or not a consumer will leave their current service.

**2. Which ones?**

According to the summary output provided by statsmodels, the following independent variables in my regression do not meet the criteria of being statistically significant:
* **Gender_Male** (p-value = 0.5727)
* **Device Protection_Yes** (p-value = 0.5419)
* **Internet Service_Fiber optic** (p-value = 0.4920)
* **Senior Citizen_Yes** (p-value = 0.3496)
* **Internet Service_No** (p-value = 0.2872)
* **Online Backup_Yes** (p-value = 0.1659)

8. Re-train the model using just the statistically significant predictors.


In [20]:
# 1. Define the list of insignificant predictors
insignificant_cols = [
    'Gender_Male',
    'Device Protection_Yes',
    'Internet Service_Fiber optic',
    'Senior Citizen_Yes',
    'Internet Service_No',
    'Online Backup_Yes'
]

# 2. Drop these columns from all three partitions
X_train_sig = X_train.drop(columns=insignificant_cols)
X_val_sig = X_val.drop(columns=insignificant_cols)
X_test_sig = X_test.drop(columns=insignificant_cols)

# 3. Initialize and train the NEW Logistic Regression model
logistic_model_sig = LogisticRegression(max_iter=1000, random_state=42)
logistic_model_sig.fit(X_train_sig, y_train)

# 4. Verify the changes
print("--- Re-training Successful ---")
print(f"Original number of features: {X_train.shape[1]}")
print(f"New number of features: {X_train_sig.shape[1]}")
print("\nModel trained exclusively on statistically significant predictors.")

--- Re-training Successful ---
Original number of features: 18
New number of features: 12

Model trained exclusively on statistically significant predictors.


9. Check model performance again by partition with total accuracy, precision, recall, F1 score, and AUC.


In [21]:
# ---------------------------------------------------------
# STEP 1: GENERATE NEW PREDICTIONS
# ---------------------------------------------------------
# Hard class predictions using the new parsimonious model
y_train_pred_sig = logistic_model_sig.predict(X_train_sig)
y_val_pred_sig = logistic_model_sig.predict(X_val_sig)
y_test_pred_sig = logistic_model_sig.predict(X_test_sig)

# Probabilities using the new parsimonious model
y_train_proba_sig = logistic_model_sig.predict_proba(X_train_sig)[:, 1]
y_val_proba_sig = logistic_model_sig.predict_proba(X_val_sig)[:, 1]
y_test_proba_sig = logistic_model_sig.predict_proba(X_test_sig)[:, 1]

# ---------------------------------------------------------
# STEP 2: EVALUATE THE NEW PREDICTIONS
# ---------------------------------------------------------
# TRAINING SET EVALUATION
print("--- Classification Report - Training Set (Parsimonious) ---")
print(classification_report(y_train, y_train_pred_sig))
train_auc_sig = roc_auc_score(y_train, y_train_proba_sig)
print(f"Training AUC: {train_auc_sig:.4f}\n")

# VALIDATION SET EVALUATION
print("--- Classification Report - Validation Set (Parsimonious) ---")
print(classification_report(y_val, y_val_pred_sig))
val_auc_sig = roc_auc_score(y_val, y_val_proba_sig)
print(f"Validation AUC: {val_auc_sig:.4f}\n")

# TEST SET EVALUATION
print("--- Classification Report - Test Set (Parsimonious) ---")
print(classification_report(y_test, y_test_pred_sig))
test_auc_sig = roc_auc_score(y_test, y_test_proba_sig)
print(f"Test AUC: {test_auc_sig:.4f}\n")

--- Classification Report - Training Set (Parsimonious) ---
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      3104
           1       0.67      0.59      0.63      1121

    accuracy                           0.81      4225
   macro avg       0.77      0.74      0.75      4225
weighted avg       0.81      0.81      0.81      4225

Training AUC: 0.8606

--- Classification Report - Validation Set (Parsimonious) ---
              precision    recall  f1-score   support

           0       0.85      0.91      0.88      1061
           1       0.64      0.52      0.57       348

    accuracy                           0.81      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409

Validation AUC: 0.8493

--- Classification Report - Test Set (Parsimonious) ---
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1009
           1 

**1. How did performance change?**

The performance changed very minimally between the full model and the parsimonious model.
* Overall accuracy remained identical across all partitions (80% on the Test set).
* The Test AUC experienced a microscopic drop from 0.8528 in the original model to 0.8510 in the new model.
* The model's ability to predict the churn class specifically (Class 1) saw a very slight reduction, with the Test recall dropping from 0.58 to 0.56, and the Test F1-score dropping from 0.63 to 0.61.

**2. Which model is better?**

The **parsimonious model (the second model)** is strictly better.

Even though there was an incredibly tiny drop in performance metrics (fractions of a percent), we achieved nearly identical predictive power while completely removing 6 complex variables from our equation. Simpler is always better if performance is comparable.

The trade-off of losing ~0.0018 AUC in exchange for a significantly more efficient and streamlined model is an absolute win.

11. Re-train the model with the addition of an interaction term.  What variables did you combine and why?  Is the new interaction term statistically significant?

In [22]:
# ---------------------------------------------------------
# STEP 1: CREATE THE INTERACTION TERM
# ---------------------------------------------------------
# We use .copy() to avoid SettingWithCopy warnings on our parsimonious datasets
X_train_interact = X_train_sig.copy()
X_val_interact = X_val_sig.copy()
X_test_interact = X_test_sig.copy()

# Create the Tenure * Monthly Charges interaction term
X_train_interact['Tenure_x_MonthlyCharges'] = X_train_interact['Tenure Months'] * X_train_interact['Monthly Charges']
X_val_interact['Tenure_x_MonthlyCharges'] = X_val_interact['Tenure Months'] * X_val_interact['Monthly Charges']
X_test_interact['Tenure_x_MonthlyCharges'] = X_test_interact['Tenure Months'] * X_test_interact['Monthly Charges']

# ---------------------------------------------------------
# STEP 2: TRAIN THE LOGISTIC REGRESSION MODEL
# ---------------------------------------------------------
logistic_model_interact = LogisticRegression(random_state=42, max_iter=1000)
logistic_model_interact.fit(X_train_interact, y_train)

print("Logistic Regression model with interaction term trained successfully.\n")

# ---------------------------------------------------------
# STEP 3: CHECK STATISTICAL SIGNIFICANCE (STATSMODELS)
# ---------------------------------------------------------
# Convert to float to be safe and add a constant for statsmodels
X_train_sm_interact = X_train_interact.astype(float)
X_train_sm_interact = sm.add_constant(X_train_sm_interact)

# Fit the statsmodels logit model
logit_model_interact = sm.Logit(y_train, X_train_sm_interact)
result_interact = logit_model_interact.fit(disp=0) # disp=0 hides the iteration output

# Extract and display the p-value for our new interaction term
interaction_p_value = result_interact.pvalues['Tenure_x_MonthlyCharges']
print(f"--- Statistical Significance ---")
print(f"P-value for Tenure_x_MonthlyCharges: {interaction_p_value:.6f}")

Logistic Regression model with interaction term trained successfully.

--- Statistical Significance ---
P-value for Tenure_x_MonthlyCharges: 0.000206


**1. What variables did you combine and why?**

I created an interaction term by combining **Tenure Months** and **Monthly Charges** (`Tenure_x_MonthlyCharges`).

I chose this combination because logically, the impact of a high monthly bill on a customer's decision to churn depends heavily on how long they have been with the company. A brand-new customer might be highly sensitive to a $100/month charge and leave immediately. Conversely, a customer who has been with the company for 60 months is already accustomed to paying that rate, meaning the `Monthly Charges` variable has a different behavioral impact depending on the `Tenure Months` variable.

**2. Is the new interaction term statistically significant?**

Yes, it is highly statistically significant. By running the updated data through `statsmodels`, the calculated p-value for the new `Tenure_x_MonthlyCharges` interaction term is **0.000206**. Because this value is well below the standard alpha threshold of 0.05, we have strong mathematical evidence that the interaction between a customer's tenure and their monthly bill plays a significant role in predicting their likelihood of churning.

12. Check model performance again by partition with total accuracy, precision, recall, F1 score, and AUC.


In [25]:
# ---------------------------------------------------------
# STEP 1: GENERATE PREDICTIONS FOR THE INTERACTION MODEL
# ---------------------------------------------------------
# Hard class predictions (1 or 0)
y_train_pred_interact = logistic_model_interact.predict(X_train_interact)
y_val_pred_interact = logistic_model_interact.predict(X_val_interact)
y_test_pred_interact = logistic_model_interact.predict(X_test_interact)

# Predicted probabilities (percentage likelihood)
y_train_proba_interact = logistic_model_interact.predict_proba(X_train_interact)[:, 1]
y_val_proba_interact = logistic_model_interact.predict_proba(X_val_interact)[:, 1]
y_test_proba_interact = logistic_model_interact.predict_proba(X_test_interact)[:, 1]

# ---------------------------------------------------------
# STEP 2: EVALUATE THE NEW PREDICTIONS
# ---------------------------------------------------------
print("--- Classification Report - Training Set (Interaction) ---")
print(classification_report(y_train, y_train_pred_interact))
train_auc_interact = roc_auc_score(y_train, y_train_proba_interact)
print(f"Training AUC: {train_auc_interact:.4f}\n")

print("--- Classification Report - Validation Set (Interaction) ---")
print(classification_report(y_val, y_val_pred_interact))
val_auc_interact = roc_auc_score(y_val, y_val_proba_interact)
print(f"Validation AUC: {val_auc_interact:.4f}\n")

print("--- Classification Report - Test Set (Interaction) ---")
print(classification_report(y_test, y_test_pred_interact))
test_auc_interact = roc_auc_score(y_test, y_test_proba_interact)
print(f"Test AUC: {test_auc_interact:.4f}\n")

--- Classification Report - Training Set (Interaction) ---
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      3104
           1       0.67      0.59      0.63      1121

    accuracy                           0.82      4225
   macro avg       0.77      0.74      0.75      4225
weighted avg       0.81      0.82      0.81      4225

Training AUC: 0.8624

--- Classification Report - Validation Set (Interaction) ---
              precision    recall  f1-score   support

           0       0.85      0.91      0.88      1061
           1       0.64      0.51      0.57       348

    accuracy                           0.81      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.80      0.81      0.80      1409

Validation AUC: 0.8496

--- Classification Report - Test Set (Interaction) ---
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1009
           1    

**1. Did adding the interaction term improve your model?**

Yes, adding the interaction term technically improved the model's overall predictive power, although the improvement was marginal.

**2. Why or why not?**

We can see this improvement mathematically in two ways:

* **The P-Value:** The interaction term (`Tenure_x_MonthlyCharges`) had a highly significant p-value of 0.0002, proving the algorithm found genuine mathematical value in the relationship between those two variables.

* **The AUC Score:** The Test AUC increased slightly from **0.8510** (parsimonious model) to **0.8516** (interaction model). The Training and Validation AUCs also saw minor bumps. AUC measures how well the model separates the probabilities of the two classes, and the slight increase proves the model is slightly more confident in its probability calculations.

**However, from a practical business perspective:**

The hard classification metrics at the default 0.5 threshold—Test Accuracy (80%), Class 1 Precision (67%), Class 1 Recall (56%), and Class 1 F1-score (61%)—remained completely identical. The interaction term made the model's underlying probability calculations more accurate, but the shift was not large enough to push any new customers over the 50% threshold to change their final "Yes/No" churn classification.

14. Make a final choice of which of the three models is best?  Justify your response.


The **Interaction Model (Model 3)** is the best choice for this dataset.

**Justification:**
Choosing the best model requires balancing predictive power with parsimony (simplicity).

1. **Why it beats the Full Model (Model 1):**

The Full Model had a higher Test AUC of 0.8528 compared to the Reduced Model at 0.8516 but it used 6 variables which we found were not statistically significant (p-value > 0.05). Using 6 unnecessary data points to generate a small change in AUC for millions of customers is an inefficient use of resources and increases the risk of overfitting to future observations.

2. **Why it beats the Parsimonious Model (Model 2):**

The Parsimonious model effectively reduced the complexity of the equation through the removal of the noise but yielded an overall increase in the test area under the ROC curve (AUC) to 0.8510. Upon adding an interaction term between Tenure Months and Monthly Charges to Model 3, we were able to model a critical aspect of actual business (the longer a customer has been with an organization, the more tolerant they are to changes on their bill) and this addition alone increased our test AUC back to 0.8516, without adding extraneous variables to clutter the model.

**Final Conclusion:**

The Interaction Model is highly interpretable, mathematically sound, free of statistical noise, and accurately captures the complex relationship between a customer's tenure, their monthly bill, and their likelihood to churn.